<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Appendix E · Affine Term-Structure Models

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/appx_e_affine_term_structure_models.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

Affine term-structure models give Vasicek and Cox-Ingersoll-Ross their closed-form bond prices. After working through this companion, you can:

- state the affine conditions on drift and variance that keep zero-coupon bond pricing tractable;
- compute Vasicek bond prices from the functions A(t,T) and B(t,T) with the short rate entering linearly;
- translate bond prices into continuously compounded zero rates and read the effect of mean reversion in the loading B(t,T);
- explain why calibration residuals and parameter checks matter as much as the pricing formula itself.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Connect the appendix to a calculation

Chapter 12 fits Vasicek, CIR, and Hull-White dynamics to market curves, and Chapter 23 verifies such fits through repricing and residual checks. This appendix shows where the exponential-affine bond-price form comes from: once the drift and instantaneous variance are affine in the short rate, the bond-pricing problem collapses into two ordinary differential equations for A(t,T) and B(t,T), and the bond price becomes the exponential of A(t,T) minus B(t,T) times the current short rate.

The companion figure script `appx_e_affine_yield_curves.py` shows the smooth term-structure families this parametric structure generates, with curve shape controlled by a small parameter set rather than by node-by-node fitting.

In [ ]:
import pandas as pd
figure_scripts = sorted((ROOT / "code" / "figures").glob("appx_e_*.py"))
pd.DataFrame({"figure_script": [p.name for p in figure_scripts]})

## Review prompts

- Write the exponential-affine bond-price form and state which of the two functions measures sensitivity to the current short rate.
- For the Vasicek loading B(t,T), explain why it approaches 1/kappa for long maturities and what faster mean reversion implies for long-bond rate sensitivity.
- Convert the mini-lab bond prices into continuously compounded zero rates and check that each rate sits between the 4% short rate and the 3% long-run level.
- Name two diagnostics, beyond the root mean squared yield error, that should accompany a fitted affine model.

## Mini-lab: Vasicek bond prices from A and B

The mini-lab evaluates the Vasicek bond-price template with kappa=0.50, theta=0.03, sigma=0.01, and a current short rate of 4% under the risk-neutral measure. Times to maturity are 1, 3, and 5 years. The loading B follows from (1 minus exp of minus kappa times tau) over kappa, and the level function A follows from the standard Vasicek solution including the variance adjustment to the drift. Zero rates are continuously compounded and annualized. All inputs are illustrative.

In [ ]:
import numpy as np
import pandas as pd

kappa, theta, sigma, r0 = 0.50, 0.03, 0.01, 0.04  # Vasicek parameters
tau = np.array([1.0, 3.0, 5.0])  # times to maturity in years
B = (1 - np.exp(-kappa * tau)) / kappa  # short-rate loading
var_adj = sigma ** 2 / (2 * kappa ** 2)  # drift compensation from variance
A = (theta - var_adj) * (B - tau) - sigma ** 2 * B ** 2 / (4 * kappa)
price = np.exp(A - B * r0)  # exponential-affine bond price
zero = -np.log(price) / tau  # continuously compounded zero rates
pd.DataFrame({"tau": tau, "B": B.round(4), "A": A.round(4),
              "price": price.round(4), "zero_rate": zero.round(4)})

### What to observe

B rises from 0.7869 to 1.8358 and approaches the cap 1/kappa=2. The bond prices are 0.9629, 0.9000, and 0.8454, giving zero rates of 3.79%, 3.51%, and 3.36%. Although the short rate is 4%, mean reversion toward theta=3% damps its influence at longer maturities, which is precisely the effect the loading B captures.

## Exercises

1. **Change the reversion speed.** Set kappa to 1.00 in the mini-lab and recompute the table. Explain how the five-year zero rate moves and why. *Hint:* B(5) falls toward 1, so today's 4% short rate matters less at long maturities.
2. **Change the short rate.** Keep all mini-lab parameters fixed except the short rate, which you set to 2%. Recompute the zero rates and describe how the curve flips. *Hint:* The zero rates now rise with maturity toward the 3% long-run level from below.
3. **CIR loading.** Look up the Cox-Ingersoll-Ross function B(t,T) and explain why it still satisfies B(T,T)=0 and increases with maturity despite the square-root diffusion. *Hint:* The instantaneous variance is still linear in the state, which is what the affine pricing argument requires.
4. **Calibration residual.** Suppose a fitted Vasicek model matches the 1-year and 5-year zero rates but misses the 3-year rate by 15 basis points. Sketch two possible causes and one check for each. *Hint:* Curve-interpolation choices and calibration weights on the middle maturities are common suspects.
5. **Yield form.** Derive the identity z(t,T) = (B(t,T) r_t minus A(t,T)) divided by (T minus t) from the bond-price formula, and verify it numerically against the mini-lab output. *Hint:* Take the logarithm of the price and divide by time to maturity.

For more detail, see Appendix E of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">